Starting the notebook to visualize and test the logic implemented in bloom_filter.py

In [ ]:
import sys, os, math, random, string, time, hashlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np


sys.path.insert(0, os.getcwd())
from bloom_filter import BloomFilter

print("Imports OK")

1. What is a Bloom filter?
A Bloom filter is a probabilistic data structure backed by a bit array.

insert(item) — hash the item k times, set those k bits to 1
contains(item) — hash the item k times, check those k bits
If any bit is 0 → the item is definitely not in the set
If all bits are 1 → the item is probably in the set (false positive possible)

In [ ]:
# create a filter designed for 200 items at 1% false positive rate
demo_filter = BloomFilter(expected_items=200, false_positive_rate=0.01)
print(demo_filter)
print(f"Bit array size : {demo_filter.bit_array_size} bits")
print(f"Hash functions : {demo_filter.num_hash_functions}")

# insert some fruits
fruits = ["apple", "banana", "cherry", "date", "elderberry",
          "fig", "grape", "honeydew", "kiwi", "lemon"]
for fruit in fruits:
    demo_filter.insert(fruit)

print(f"\nInserted {len(fruits)} fruits.\n")

# all inserted fruits must come back True
print("Querying inserted fruits (all should be True):")
for fruit in fruits:
    result = demo_filter.contains(fruit)
    status = "✓" if result else "✗ ERROR"
    print(f"  {status}  {fruit}")

# words never inserted should mostly come back False
not_fruits = ["mango", "nectarine", "orange", "papaya", "quince"]
print("\nQuerying words NOT inserted (should mostly be False):")
for word in not_fruits:
    print(f"  {word}: {demo_filter.contains(word)}")

    


2. Testing the hash functions

Hash function are distributing the positions uniformly across the bit array.
We will check this for three data types here:

- Common English words
- Random strings
- DNA sequences (only characters A, C, G, T)

We will aslo compare the actual fill fraction to the theoretical expectation: 1 - e^(-k·n/m).

In [ ]:
def check_hash_distribution(items, n_expected, fpr=0.01, label=""):
    bf = BloomFilter(expected_items=n_expected, false_positive_rate=fpr)
    for item in items:
        bf.insert(item)

    bits_set = bf.count_bits_set()
    fill = bits_set / bf.bit_array_size

    k, n, m = bf.num_hash_functions, len(items), bf.bit_array_size
    expected_fill = 1 - math.exp(-k * n / m)

    print(f"{label}")
    print(f"  Items inserted : {n}")
    print(f"  Fill fraction  : {fill:.4f}  (expected {expected_fill:.4f})")
    print(f"  Difference     : {abs(fill - expected_fill):.4f}")
    print()


random.seed(0)